# Importing the neccessary Libraries

In [1]:
import os
import uuid
import base64
from dotenv import load_dotenv

#libraries for the scanned textbooks
import fitz
import pytesseract
from PIL import Image

#Langchain libraries
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.documents import Document


c:\Users\Loba\MB_ASSISTANT\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Loba\MB_ASSISTANT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

load_dotenv()

True

Let us load the scanned copies
-----------------------------------------------
PyMuPixmap to PIL
1. fitz opens the PDF and takes a "screenshot" (pixmap) of the page.
2. We convert that raw screenshot into a standard PIL Image.
3. We hand the PIL Image to Tesseract to read the words.
-----------------------------------------------------------

In [ ]:
#Instantiating pytesseract
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [4]:
# Define function to load scanned textbooks
def extract_scanned_textbooks(pdf_path):
    """
    this function uses PyMuPDf (fitz) and Tesseract to convert scanned pdfs to texts
    """
    print(f"Opening {pdf_path} with PyMuPDF...")
    doc = fitz.open(pdf_path)
    docs = []

    for page_num, page in enumerate(doc):
        if page_num % 200 == 0:
            print(f"Processed {page_num} pages...")

        # Take High Resolution Screenshot of the page (matrix scales it up for better OCR reading)
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
        
        #Convert to PIL image for Tesseract
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        #Extract text using Tesseract
        extracted_text = pytesseract.image_to_string(img)

        #keeping only pages that have text
        if extracted_text.strip():
            docs.append(
                Document(
                    page_content=extracted_text,
                    #Include source
                    metadata={
                        "source": pdf_path, 
                        "page": page_num + 1, 
                        "type": "ocr_text"
                        }
                )
            )
    
    return docs

In [5]:
# let's define a function to load all the pdfs from our scanned textbooks directory
def load_scanned_directory(directory_path):
    """
    Acts as a pdf Directory loader for the scanned OCR pdfs.
    loops through the folder and processes every pdf it finds.
    """
    all_scanned_docs = []

    for filename in os.listdir(directory_path):
        #combine folder path and file name
        full_file_path = os.path.join(directory_path, filename)
        print(f"\n Starting OCR on Textbook: {filename}")

        #use the PyMuPDF function
        book_docs = extract_scanned_textbooks(full_file_path)

        #add this book's pages  to out master list
        all_scanned_docs.extend(book_docs)
    
    return all_scanned_docs

In [ ]:
#This should work, but it will take a lot of time, so i might just try using paddle OCR on colab and use T4 GPU
# Let's load the scanned textbooks
folder = r"data\scanned_textbooks"

# Check if the variable exists in memory AND is not empty
if 'total_scanned_docs' in locals() and total_scanned_docs:
    print(f"Skipping OCR... {len(total_scanned_docs)} pages already exist in memory!")
else:
    print("Starting Batch OCR Process....")
    total_scanned_docs = load_scanned_directory(folder)
    print(f"\n Batch Complete! Extracted a total of {len(total_scanned_docs)} pages across all textbooks.")

In [ ]:
# Define exactly where you want the final text file saved
output_filepath = r"data\scanned_textbooks\extracted_board_prep.txt"

print(f"Writing {len(total_scanned_docs)} pages to local text file...")

# Open the file in write mode ('w') with utf-8 encoding to prevent character crashes
with open(output_filepath, "w", encoding="utf-8") as f:
    
    for doc in total_scanned_docs:
        # 1. Safely extract the tracking data from the metadata dictionary
        source_name = doc.metadata.get("source", "Unknown_Source")
        page_num = doc.metadata.get("page", "Unknown")
        
        # 2. Write a clear header so LangChain can chunk it easily later
        f.write(f"\n\n--- SOURCE: {source_name} | PAGE: {page_num} ---\n\n")
        
        # 3. Write the actual medical text extracted by Tesseract
        f.write(doc.page_content)

print(f"✅ Success! All OCR text saved locally to: {output_filepath}")

Let's use the multimodal method to extract the slides.
----------------------------------------------------

In [30]:
import sys
import os

# Adds the parent directory (MB_ASSISTANT) to the system path
sys.path.append(os.path.abspath('..'))

In [6]:
from langchain_core.messages import HumanMessage
from src.config import vision_model, llm

vision_llm = ChatGroq(
    model = vision_model,
    api_key = os.getenv("GROQ_API_KEY"),
    temperature = 0,
    max_retries=3
)

In [7]:
#define fuction to summarize images/diagrams
from src.prompts import VISION_LLM_PROMPT

def summarize_medical_diagram(image_bytes):
    """
    Sends the extracted image to Langchain's ChatGroq (vision_llm) for summary.
    """
    base64_img = base64.b64encode(image_bytes).decode('utf-8')

    try:
        #multimodal human message
        message = HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": VISION_LLM_PROMPT
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_img}"
                    }
                }
            ]
        )
        response = vision_llm.invoke([message])
        return response.content
    
    except Exception as e:
        print(f"Vision Model Error: {e}")
        return "Image description failed due to API error"

Have to skip this because of Groq's token limit.. 
Switching to a page by page processing, so that I can continue from where I stop or where limit was reached

In [8]:
import time
#Define slides processing function
def process_slides(pdf_path):
    """
    Extracts both the text and Langchain-summarized images from a slide deck.
    """
    print(f"\n Opening Slide Deck: {os.path.basename(pdf_path)}")
    doc = fitz.open(pdf_path)
    slide_docs = []

    for page_num, page in enumerate(doc):
        #extract text 
        page_text = page.get_text()
        if page_text.strip():
            slide_docs.append(
                Document(
                    page_content=page_text,
                    metadata={
                        "source": os.path.basename(pdf_path),
                        "page": page_num +1,
                        "type": "slide_text"
                    }
                )
            )
        
        #Extract images and pass to model
        image_list = page.get_images(full=True)
        if image_list:
            if page_num % 100 == 0:
                print(f" found {len(image_list)} image(s) on page {page_num + 1}. Calling Vision AI")

            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                #call summarize function
                image_description = summarize_medical_diagram(image_bytes)

                #append to slide docs
                slide_docs.append(
                    Document(
                        page_content=f"IMAGE DESCRIPTION: {image_description}",
                        metadata={"source": os.path.basename(pdf_path), "page": page_num + 1, "type": "slide_image"}
                    )
                )

                #delay api calls for 2 seconds
                time.sleep(2)

In [9]:
#processing for each file in the dorectory
def batch_process_slides(directory_path, output_filepath):
    """
    Processes slides incrementally. Safe to stop and restart at any time.
    """
    all_slide_docs = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".pdf"):
            full_path = os.path.join(directory_path, filename)
            docs = process_slides(full_path)
            all_slide_docs.extend(docs)

    os.makedirs9os.path.dirname(output_filepath, exist_ok=True)
    with open(output_filepath, "w", encoding="utf-8") as f:
        for doc in all_slide_docs:
            source = doc.metadata.get("source", "Unknow_Slide")
            page = doc.metadata.get("page", "Unknown")
            doc_type = doc.metadata.get("type", "Unknown_Type")

            f.write(f"\n\n--- SOURCE: {source} | PAGE: {page} | TYPE: {doc_type} ---\n\n")
            f.write(doc.page_content)
        
    print(f"\n Success! All multimodal slide data are saved to: {output_filepath}")
    return all_slide_docs

In [ ]:
#get all the slides as txt
slides_folder = r"C:\Users\Loba\MB_ASSISTANT\data\slides"
output_text_file = r"C:\Users\Loba\MB_ASSISTANT\data\slidesextracted_slides.txt"

if os.path.exists(output_text_file):
    print(f"Skipping extraction. File already exists at: {output_text_file}")
else:
    print("Starting Multimodal Slide Extraction with LangChain...")
    extracted_slides = batch_process_slides(slides_folder, output_text_file)

# Page by Page Processing
Due to free tier token limits, there is a need to save in batches so that you can pick up from last saved batch

In [7]:
import time

def get_page_level_progress(output_filepath):
    """
    Scans the text file and finds the highest page completed for each pdf.
    """
    progress = {}
    if os.path.exists(output_filepath):
        with open(output_filepath, "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("--- SOURCE:"):
                    parts = line.split("|")
                    if len(parts) >= 2:
                        source_part = parts[0].replace("--- SOURCE:", "").strip()
                        page_part = parts[1].replace("PAGE:", "").strip()
                        try:
                            page_num = int(page_part)
                            if source_part not in progress or page_num > progress[source_part]:
                                progress[source_part] = page_num
                        except ValueError:
                            pass
    return progress


In [8]:
#Process and saves slides per page

def process_and_save_slides(pdf_path, output_filepath, start_page=0):
    """
    Processes a slide deck and saves EVERY SINGLE PAGE to disk instantly.
    """
    filename = os.path.basename(pdf_path)
    print(f"\n Opening Slide Deck: {filename} (Resuming from page {start_page + 1})")

    doc = fitz.open(pdf_path)

    for page_num in range(start_page, len(doc)):
        page = doc[page_num]
        current_page_docs = []

        #Extract slide
        page_text = page.get_text()
        if page_text.strip():
            current_page_docs.append(
                Document(
                    page_content=page_text,
                    metadata={
                        "source": filename,
                        "page": page_num + 1,
                        "type": "slide_text"
                    }
                )
            )

        #Extract images and pass to model
        image_list = page.get_images(full=True)
        if image_list:
            if page_num % 100 == 0:
                print(f" found {len(image_list)} image(s) on page {page_num + 1}. Calling Vision AI")

            for img_index, img in enumerate(image_list):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                #call summarize function
                image_description = summarize_medical_diagram(image_bytes)

                current_page_docs.append(
                    Document(
                        page_content=f"IMAGE DESCRIPTION: {image_description}",
                        metadata={
                            "source": filename,
                            "page": page_num + 1,
                            "type": "slide_image"
                        }
                    )
                )

                time.sleep(2)
        
        if current_page_docs:
            with open(output_filepath, "a", encoding="utf-8") as f:
                for d in current_page_docs:
                    source = d.metadata.get("source", "Unknown_Slide")
                    page_meta = d.metadata.get("page", "Unknown")
                    doc_type = d.metadata.get("type", "Unknown_Type")
                    
                    f.write(f"\n\n--- SOURCE: {source} | PAGE: {page_meta} | TYPE: {doc_type} ---\n\n")
                    f.write(d.page_content)
            if page_num % 100 == 0:        
                print(f" Saved Page {page_num + 1} to disk.")



In [9]:
def batch_process_slides(directory_path, output_filepath):
    """
    Manages the folder and handles the resume logic.
    """
    os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    
    # Figure out exactly which page we stopped at for every file
    progress_dict = get_page_level_progress(output_filepath)
    
    for filename in os.listdir(directory_path):
        if filename.endswith(".pdf"):
            full_path = os.path.join(directory_path, filename)
            
            # Find the last completed page (default to 0 if it's a new file)
            last_page = progress_dict.get(filename, 0)
            
            # Check if the file is 100% complete
            doc = fitz.open(full_path)
            total_pages = len(doc)
            doc.close()
            
            if last_page >= total_pages:
                print(f"⏩ Skipping {filename} (All {total_pages} pages already processed)")
                continue
                
            # Process starting from the next unprocessed page
            process_and_save_slides(full_path, output_filepath, start_page=last_page)
            
    print(f"\n Batch Complete! All slide data safely stored at: {output_filepath}")

In [ ]:
# Ensure you use your absolute paths here
slides_folder = r"C:\Users\Loba\MB_ASSISTANT\data\slides"
output_text_file = r"C:\Users\Loba\MB_ASSISTANT\data\slides\extracted_slides.txt"

print("Starting Micro-Checkpoint Slide Extraction...")
batch_process_slides(slides_folder, output_text_file)

Starting Micro-Checkpoint Slide Extraction...

 Opening Slide Deck: O and G 1 Merged.pdf (Resuming from page 1)
 found 1 image(s) on page 1. Calling Vision AI
 Saved Page 1 to disk.
 found 30 image(s) on page 101. Calling Vision AI
 Saved Page 101 to disk.


# Trying LLAMAINDEX to extract

In [6]:
import os
import json
import asyncio
from pypdf import PdfReader, PdfWriter
from llama_parse import LlamaParse
from dotenv import load_dotenv

load_dotenv()

# We upgrade the function to 'async def' to natively handle the event loop
async def process_with_llamaparse(input_dir, output_dir, batch_size=20):
    os.makedirs(output_dir, exist_ok=True)
    progress_file = os.path.join(output_dir, "parse_progress.json")
    
    if os.path.exists(progress_file):
        with open(progress_file, "r") as f:
            progress = json.load(f)
    else:
        progress = {}

    # Initialize parser
    parser = LlamaParse(
        api_key=os.getenv("LLAMA_CLOUD_API_KEY"),
        result_type="markdown",
        verbose=False
    )

    for filename in os.listdir(input_dir):
        if not filename.endswith(".pdf"):
            continue
            
        filepath = os.path.join(input_dir, filename)
        output_md = os.path.join(output_dir, f"{filename.replace('.pdf', '')}.md")
        
        reader = PdfReader(filepath)
        total_pages = len(reader.pages)
        
        start_page = progress.get(filename, 0)
        
        if start_page >= total_pages:
            print(f"⏩ Skipping {filename} (Already 100% processed)")
            continue
            
        print(f"\n📂 Processing {filename} (Resuming from page {start_page + 1}/{total_pages})")
        
        while start_page < total_pages:
            end_page = min(start_page + batch_size, total_pages)
            print(f"   ⏳ Sending pages {start_page + 1} to {end_page} to LlamaCloud...")
            
            writer = PdfWriter()
            for i in range(start_page, end_page):
                writer.add_page(reader.pages[i])
                
            temp_pdf = "temp_llamaparse_chunk.pdf"
            with open(temp_pdf, "wb") as f:
                writer.write(f)
            
            try:
                # THE UPGRADE: We 'await' the pure async function. No more closed event loops.
                parsed_docs = await parser.aload_data(temp_pdf)
                
                # THE FAILSAFE: If the API chokes and returns empty data, do NOT advance.
                if not parsed_docs:
                    print("   ⚠️ LlamaCloud returned empty data. Retrying in 5 seconds...")
                    await asyncio.sleep(5)
                    continue

                # Ensure we only join strings that actually have content
                chunk_text = "\n\n".join([doc.text for doc in parsed_docs if doc.text])
                
                with open(output_md, "a", encoding="utf-8") as f:
                    f.write(f"\n\n\n\n")
                    f.write(chunk_text)
                    
                start_page = end_page
                progress[filename] = start_page
                
                with open(progress_file, "w") as f:
                    json.dump(progress, f)
                    
                print(f"   ✅ Saved. Progress: {start_page}/{total_pages}")
                await asyncio.sleep(1)
                
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "limit" in error_msg or "quota" in error_msg or "exceeded" in error_msg:
                    print("\n🛑 DAILY LLAMAPARSE LIMIT REACHED (1,000 pages).")
                    print(f"   Progress securely saved at page {start_page}. See you tomorrow!")
                    if os.path.exists(temp_pdf): os.remove(temp_pdf)
                    return 
                else:
                    print(f"   ❌ Network Error: {e}. Retrying in 10s...")
                    await asyncio.sleep(10)
                    
    if os.path.exists("temp_llamaparse_chunk.pdf"):
        os.remove("temp_llamaparse_chunk.pdf")
    print("\n🎉 ALL PDFS PROCESSED SUCCESSFULLY!")

# --- EXECUTION ---
raw_pdfs_folder = r"C:\Users\Loba\MB_ASSISTANT\data\slides" 
markdown_output_folder = r"C:\Users\Loba\MB_ASSISTANT\data\markdown_vault"

# Because Jupyter natively runs an async loop, we use 'await' right here
await process_with_llamaparse(raw_pdfs_folder, markdown_output_folder, batch_size=20)

C:\Users\Loba\AppData\Local\Temp\ipykernel_8740\3843872606.py:5: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


⏩ Skipping O and G 1 Merged.pdf (Already 100% processed)
⏩ Skipping Paediatrics CS edited.pdf (Already 100% processed)

🎉 ALL PDFS PROCESSED SUCCESSFULLY!


In [9]:
# --- EXECUTION ---
raw_pdfs_folder = r"C:\Users\Loba\MB_ASSISTANT\data\scanned_textbooks" 
markdown_output_folder = r"C:\Users\Loba\MB_ASSISTANT\data\markdown_vault"

# Because Jupyter natively runs an async loop, we use 'await' right here
await process_with_llamaparse(raw_pdfs_folder, markdown_output_folder, batch_size=20)

⏩ Skipping AZUBUIKE.pdf (Already 100% processed)

📂 Processing Textbook-of-Obstetrics-and-Gynaecology-for-Medical-Students-2nd-Edition-Akin-Agboola-2006.pdf (Resuming from page 501/559)
   ⏳ Sending pages 501 to 520 to LlamaCloud...
   ✅ Saved. Progress: 520/559
   ⏳ Sending pages 521 to 540 to LlamaCloud...
   ✅ Saved. Progress: 540/559
   ⏳ Sending pages 541 to 559 to LlamaCloud...
   ✅ Saved. Progress: 559/559

🎉 ALL PDFS PROCESSED SUCCESSFULLY!


# Set up Pinecone and Ingest Data

In [3]:
# Get pinecone variables
load_dotenv()
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME")

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


In [13]:
#get embedding model

def download_embedding_model():
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5", 
    encode_kwargs={'normalize_embeddings': True}
    )
    return embeddings

In [14]:
embeddings = download_embedding_model()

C:\Users\Loba\AppData\Local\Temp\ipykernel_20380\4181710103.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 434.31it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
len(embeddings.embed_query("hi"))

1024

In [18]:
#markdown loader
markdown_folder = r"C:\Users\Loba\MB_ASSISTANT\data\markdown_vault"

raw_documents = []

#loop through folder for .md file
for filename in os.listdir(markdown_folder):
    if filename.endswith(".md"):
        filepath = os.path.join(markdown_folder, filename)

        with open(filepath, "r", encoding="utf-8") as file:
            content = file.read()

            raw_documents.append(
                {
                    "text": content,
                    "source": filename
                }
            )
print(f"✅ Successfully loaded {len(raw_documents)} Markdown files into memory.")

✅ Successfully loaded 4 Markdown files into memory.


In [22]:
#Split on headers to make semantic search more robust
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
# Fallback splitter in case a textbook paragraph is too long
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

final_chunks = []
for doc in raw_documents:
    #split by header
    header_splits = markdown_splitter.split_text(doc["text"])

    sized_splits = text_splitter.split_documents(header_splits)

    #inject source
    for split in sized_splits:
        split.metadata["source"] = doc["source"]
    
    final_chunks.extend(sized_splits)

print(f"✅ Sliced the textbooks into {len(final_chunks)} RAG-ready chunks.")

✅ Sliced the textbooks into 12468 RAG-ready chunks.


In [ ]:
#initialize pinecone
docsearch = PineconeVectorStore.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    index_name="mb-assistant"
)

# Define Agent tools

In [45]:
from pinecone import Pinecone

In [46]:
pinecone_api_key = os.getenv("PINECONE_API_KEY")
pinecone_index_name = os.getenv("PINECONE_INDEX_NAME")

In [47]:
#Vectorestore
vectorstore = PineconeVectorStore(
    index_name = pinecone_index_name,
    embedding=embeddings
)

In [4]:
from langchain_core.tools import tool

In [63]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class DiseaseInput(BaseModel):
    disease_name: str = Field(
        description="The specific medical disease or condition to search for."
    )

@tool(args_schema=DiseaseInput)
def search_disease_profile(disease_name: str) -> str:
    """
    Use this tool ONLY when the user asks for comprehensive details about a specific DISEASE, SYNDROME, or OBSTETRIC COMPLICATION.
    """
    # 1. Break the giant query into three targeted sniper shots
    q_patho = f"{disease_name} pathophysiology aetiology signs symptoms clinical features"
    q_invest = f"{disease_name} investigations diagnosis labs imaging criteria"
    q_manage = f"{disease_name} management treatment surgery drugs prognosis complications"
    
    # 2. Execute 3 isolated searches (Pulling the top 2 for each category)
    # This prevents a Table of Contents from dominating the results
    docs_patho = vectorstore.similarity_search(q_patho, k=2)
    docs_invest = vectorstore.similarity_search(q_invest, k=2)
    docs_manage = vectorstore.similarity_search(q_manage, k=2)
    
    # 3. Combine the results
    all_docs = docs_patho + docs_invest + docs_manage
    
    # 4. Deduplicate (In case the same chunk scored high in multiple queries)
    unique_docs = {doc.page_content: doc for doc in all_docs}.values()
    
    if not unique_docs:
        return "NO_DATA_FOUND"
    
    return "\n\n".join([f"[SOURCE: {d.metadata.get('source', 'Unknown File')}]\n{d.page_content}" for d in unique_docs])

print("✅ Multi-Query Disease Profiler Online.")

✅ Multi-Query Disease Profiler Online.


In [64]:
response = search_disease_profile.invoke("Retinoblastoma")
print(response)

[SOURCE: AZUBUIKE.md]
Retinoblastoma is a common primary ocular malignancy of childhood in the tropics (Table 75.1). It is a relatively slow-growing tumour and may be contained in the eye for months to years before causing grotesque proptosis. Because of late presentation in the tropics, most patients are either blind or can barely see when first seen. The median age is 11 months for those with bilateral tumours (30%) and about 23 months in unilateral tumours (70%). There is a genetic predisposition to retinoblastoma, of the autosomal dominant type with almost complete penetrance but the abnormal mutant allele is recessive. The retinoblastoma locus is the chromosome 13 and a deletion on the long arm of this chromosome (13q-syndrome) has been found. Prenatal diagnosis is now possible from the amniotic fluid.

[SOURCE: AZUBUIKE.md]
1. Abramson DH. Retinoblastoma: Diagnosis and management. Cancer, 1982:32:130-140.
2. Kodilinye HC. Retinoblastoma, in Nigeria; problems of treatment AM J.Oph

In [65]:
@tool
def search_general_medical_fact(query: str) -> str:
    """
    Use this tool FIRST for general medical questions, statistics, anatomical facts, or single data points (e.g., 'commonest cause of childhood cancer', 'normal fetal heart rate').
    """
    docs = vectorstore.similarity_search(query, k=6)
    
    if not docs:
        return "NO_DATA_FOUND"
    
    return "\n\n".join([f"[SOURCE: {d.metadata.get('source', 'Unknown File')}]\n{d.page_content}" for d in docs])

In [66]:
response = search_general_medical_fact.invoke("Retinoblastoma")
print(response)

[SOURCE: AZUBUIKE.md]
Retinoblastoma is a common primary ocular malignancy of childhood in the tropics (Table 75.1). It is a relatively slow-growing tumour and may be contained in the eye for months to years before causing grotesque proptosis. Because of late presentation in the tropics, most patients are either blind or can barely see when first seen. The median age is 11 months for those with bilateral tumours (30%) and about 23 months in unilateral tumours (70%). There is a genetic predisposition to retinoblastoma, of the autosomal dominant type with almost complete penetrance but the abnormal mutant allele is recessive. The retinoblastoma locus is the chromosome 13 and a deletion on the long arm of this chromosome (13q-syndrome) has been found. Prenatal diagnosis is now possible from the amniotic fluid.

[SOURCE: AZUBUIKE.md]
1. Abramson DH. Retinoblastoma: Diagnosis and management. Cancer, 1982:32:130-140.
2. Kodilinye HC. Retinoblastoma, in Nigeria; problems of treatment AM J.Oph

In [9]:
#internet search too

ddg_search = DuckDuckGoSearchRun()

@tool
def search_internet(query: str) -> str:
    """
    Use this ONLY if the mbbs_database lacks the answer, or if the user asks for
    recent news, current guidelines, or non-medical facts.
    """
    return ddg_search.invoke(query)

In [54]:
response = search_internet.invoke("Retinoblastoma")
print(response)

Retinoblastomais the most intrusive intraocular cancer among children.Retinoblastomais extremely rare as there are only about 200 to 300 cases every year in the United States. Retinoblastoma is the most common primary intraocular malignancy in children, a rare embryonic tumor arising from immature retinal cells due to biallelic inactivation of the RB1 tumor... Información general sobre elretinoblastomaEstadios delretinoblastomaTratamiento delretinoblastomaunilateral, bilateral y cavitario Retinoblastomais a kind of eye cancer that starts as a growth of cells in the retina. The retina is the light-sensitive lining on the inside of the eye. Retinoblastomasare the most common intraocular neoplasm found in childhood and with modern treatment modalities, are, in most cases, curable.


In [77]:
#Pubmed search
from langchain_community.utilities.pubmed import PubMedAPIWrapper
from langchain_community.tools.pubmed.tool import PubmedQueryRun

#Pubmed Custom Wrapper
pubmed_wrapper = PubMedAPIWrapper(
    top_k_results=2,
    doc_content_chars_max=3000
)

pubmed_search = PubmedQueryRun(api_wrapper=pubmed_wrapper)

In [79]:
response = pubmed_search.invoke("Retinoblastoma Management")
print(response)

Published: 2026-02-28
Title: Multifocal cerebral and orbital neoplastic lesions in a 2-year-old child: a case report and literature review emphasizing diagnostic uncertainty and ethical decision-making.
Copyright Information: © 2026. The Author(s), under exclusive licence to Springer-Verlag GmbH Germany, part of Springer Nature.
Summary::
BACKGROUND: Multifocal cerebral and orbital neoplastic lesions in early childhood are exceptionally uncommon. Multiple lesions may occur in the context of cancer predisposition syndromes or prior irradiation; in our patient, no syndromic stigmata or prior cranial irradiation were clinically evident, but a genetic predisposition could not be excluded because molecular testing was not performed.
CASE PRESENTATION: A previously healthy 2-year-old girl presented with chronic headache and progressive right-sided proptosis. MRI and CT identified six anatomically distinct lesions: an intraocular retinoblastoma with orbital extension (histologically confirmed

In [42]:
from pydantic import BaseModel, Field

class QuizInput(BaseModel):
    topic: str = Field(
        description="The specific Paediatrics or O&G disease to quiz the user on."
    )
    num: int = Field(description="The exact number of questions. Convert English words to integers."
        
    )

@tool(args_schema=QuizInput)
def generate_clinical_quiz(topic: str, num: int) -> str:
    """
    Use this when the user explicitly asks to be tested, quizzed, or wants practice questions on an MB3 topic.
    """
    #Use one of the tiils above to retuen context
    context = search_disease_profile.invoke({"disease_name": topic})

    if context == "NO_DATA_FOUND":
        return "INSTRUCTION TO AGENT: Tell the user you cannot generate a quiz because the topic is not in the verified MB3 database."
    return f"Generate {num} difficult, MB3-board-level multiple choice questions based strictly on this verified data. Provide the correct answers and brief rationales at the end:\n\n{context}"

In [73]:
toolkit = [search_disease_profile, search_general_medical_fact, search_internet, generate_clinical_quiz, pubmed_search]
print("✅ Tools loaded.")

✅ Tools loaded.


In [33]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

In [27]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.5
)

In [28]:
memory = MemorySaver()

In [35]:
from src.prompts import system_prompt

agent_executor = create_agent(
    model=llm,
    tools=toolkit,
    system_prompt=system_prompt,
    checkpointer=memory
)

In [ ]:

config = {"configurable": {"thread_id": "mb3_study_session_002"}}

response = agent_executor.invoke(
    {"messages": [("user", "Tell me everything about Retinoblastoma.")]}, 
    config=config
)

print(response["messages"][-1].content)

**Definition & Epidemiology:**  
- Retinoblastoma is the most common primary intra‑ocular malignancy of childhood. It arises from malignant transformation of retinal neuroblastic cells.  
- It accounts for ~3 % of all childhood cancers but is the leading ocular tumor in the pediatric age group.  
- Median age at diagnosis is **≈ 11 months** for bilateral disease (≈ 30 % of cases) and **≈ 23 months** for unilateral disease (≈ 70 %).  
- Higher incidence is reported in tropical and low‑resource regions, where late presentation is common; many children are blind or have only minimal vision at first contact.  

**Aetiology / Pathophysiology:**  
- **Genetic basis:** Classic “two‑hit” model involving the **RB1 tumor‑suppressor gene** on chromosome 13q14.2.  
  - **Heritable (germ‑line) form:** Autosomal‑dominant inheritance with almost complete penetrance; the mutant allele is recessive at the cellular level. Children inherit one defective RB1 allele and acquire a second somatic hit, leadin